In [ ]:
"""Merge the Thailand + Philippines QC'd AirNow NetCDFs into ONE ground-obs
file

  1. align both onto the union hourly time axis (reindex; gaps -> NaN)
  2. union the variable sets (a var present in only one network is NaN for
     the other -- e.g. CO/PM only at TH, solar_radiation/precip only at PH)
  3. concatenate sites along `x`, renumber `x` to 0..N-1
  4. add a `network` coord (TH_BMA / PH_AWS) for optional domain filtering

siteid is unique across networks (e.g. '1BMA' vs 'WPF001'/'mw002'/'MOIP'),
so stations stay distinguishable after the merge.

"""

import numpy as np
import xarray as xr


In [ ]:
PRE = ("/glade/u/home/lcthompson/mm/MELODIES-MONET/docs/examples/"
       "ungridded_support/unstructured_grid_read_uxarray/asiaaq_cs_06082026/"
       "preprocessing")
TH = f"{PRE}/thai_data/thai_aws_airnow_qc.nc"
PH = f"{PRE}/philippines_data/philippines_aws_airnow_qc.nc"
OUT = f"{PRE}/asiaaq_ground_airnow_qc.nc"
TEMPLATE_VAR = "temperature" 

def load(path, network):
    ds = xr.open_dataset(path).transpose("x", "time", ...)
    ds = ds.assign_coords(
        network=("x", np.array([network] * ds.sizes["x"], dtype=object))
    )
    return ds



In [ ]:
th = load(TH, "TH_BMA")
ph = load(PH, "PH_AWS")

# common time axis 
times = np.union1d(th["time"].values, ph["time"].values)
th = th.reindex(time=times)
ph = ph.reindex(time=times)

# union data variables; add missing as all-NaN so concat aligns
allvars = set(th.data_vars) | set(ph.data_vars)
for ds_ in (th, ph):
    for v in sorted(allvars - set(ds_.data_vars)):
        if v == "time_local":   # present in both; never fill datetime
            continue
        ds_[v] = xr.full_like(ds_[TEMPLATE_VAR], np.nan)

# concatenate sites along x, renumber x
combined = xr.concat([th, ph], dim="x", join="outer")
combined = combined.assign_coords(x=np.arange(combined.sizes["x"]))

print(combined)
print("\nsites:", combined.sizes["x"],
      "| times:", combined.sizes["time"],
      "| networks:", np.unique(combined["network"].values))
combined.to_netcdf(OUT)
print("\nwrote:", OUT)


In [ ]:

ds = xr.open_dataset("/glade/u/home/lcthompson/mm/MELODIES-MONET/docs/examples/ungridded_support/unstructured_grid_read_uxarray/asiaaq_cs_06082026/preprocessing/asiaaq_ground_airnow_qc.nc")

sid = np.asarray(ds["siteid"].values)
u, c = np.unique(sid, return_counts=True)
print("sites:", sid.size, "| unique siteid:", u.size)
print("duplicate siteids:", u[c > 1])

t = ds["time"].values
print("times:", t.size, "| unique:", np.unique(t).size)

df = ds.to_dataframe().reset_index()
print("dup (time,siteid) rows:", int(df.duplicated(subset=["time", "siteid"]).sum()))